# train3: temporary training notebook (literature methods, part 3)

Two runs of **B3**, alternating train-and-merge (`turkish_distill.alternating`): per round one arm per teacher (BGE-M3, 8K-embed) starts from the previous round's merged model and learns from its teacher only; the arms are averaged with equal weights. Budget 1: 4 rounds x 675 steps per arm (the joint run's cost); budget 2: 4 rounds x 1350 steps per arm (twice the cost, an upper bound). Every arm is an ordinary run `<name>_r<round>_<teacher>` under `runs/`, the merged model of every round is `models/<name>_r<round>` (evaluate round 4). Alternating needs the model loaded again in every arm, so it takes a few minutes more than the same number of joint steps.

All runs compare a method from the multi-teacher literature (adapted to embeddings, see `arastirma/literatur_yontemleri_plani.md`) with the plain two-teacher recipe on the same terms: 1k-base student, 200K mix, BGE-M3 + plain 8K-embed, lr 1e-4, 5400 steps (batch 128), seed 42. The reference run is `scale_two_lr1e4_mix200k_s5400` (TR-MTEB 67.16; seed 43: 67.25, so only differences above 0.3 are read). Re-running the cells after a disconnect resumes the unfinished run and skips finished ones, then the last cell releases the runtime. Delete this notebook when the results are in.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Alternating train-and-merge (Merge-of-Thought style): every round trains one arm per teacher from the model merged at the
# end of the previous round, then averages the arms. Two budgets (steps per arm = rounds * steps per round):
#   lit_b3_alt_s2700: 4 rounds x 675 steps = 2700 per arm, 5400 student steps in all = the joint run's cost;
#   lit_b3_alt_s5400: 4 rounds x 1350 steps = 5400 per arm, twice the joint run's cost (an upper bound).
# The command is idempotent: finished arms and merges are skipped, the unfinished arm resumes.
CONFIG = 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml'
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.lr=0.0001"]
ALT_RUNS = [
    # (name, rounds, steps per round)
    ('lit_b3_alt_s2700', 4, 675),
    ('lit_b3_alt_s5400', 4, 1350),
]

for name, rounds, steps in ALT_RUNS:
    overrides = ' '.join(f'--override {o}' for o in BASE)
    print(f'\n########## {name}: {rounds} rounds x {steps} steps per arm')
    get_ipython().system(f'python -m turkish_distill.alternating --config {CONFIG} --name {name} --rounds {rounds} --steps-per-round {steps} {overrides}')

print('\nEval presets (paste into an eval notebook; the merged model of the last round):')
for name, rounds, steps in ALT_RUNS:
    print(f"    '{name}': dict(model=f'{{DRIVE_ROOT}}/models/{name}_r{rounds}'),")

In [ ]:
from google.colab import runtime
runtime.unassign()